In [ ]:
import pandas as pd
import ast
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MultiLabelBinarizer
import cv2
from skimage import exposure
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cluster import MiniBatchKMeans
from collections import Counter
from matplotlib import colors as mcolors
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
import os
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import pytesseract
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
import os
import cv2
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from skimage.filters import sobel

In [ ]:
# Import the dataset and parse specific columns using custom converters
# - "Dominant_RGB" and "Hist_Features" are stored as stringified Python lists
# - "Category" is a comma-separated string, split into a list of labels
df = pd.read_csv(
    "/kaggle/input/thumbnail-features/thumbnail_visual_feature.csv",
    converters={
        "Dominant_RGB": ast.literal_eval,
        "Hist_Features": ast.literal_eval,
        "Category": lambda x: [label.strip() for label in x.split(",")],
    },
)

# Helper function to flatten a 3x3 RGB list into a 9-element array
def flatten_rgb(rgb_list):
    return np.array(rgb_list).flatten()

# Apply the flattening function to the "Dominant_RGB" column
df["Dominant_RGB_Flat"] = df["Dominant_RGB"].apply(flatten_rgb)

# Convert the flattened RGB data into a DataFrame with appropriate column names
rgb_flat_df = pd.DataFrame(
    df["Dominant_RGB_Flat"].tolist(), columns=[f"DRGB_{i}" for i in range(9)]
)

# Convert the histogram features into a separate DataFrame
hist_feat_df = pd.DataFrame(
    df["Hist_Features"].tolist(), columns=[f"Hist_{i}" for i in range(10)]
)

# Drop unnecessary columns that won't be used for modeling
df.drop(columns=["Dominant_Labels", "Dominant_RGB", "Image_Name"], inplace=True)

# One-hot encode the multi-label "Category" field
mlb_1 = MultiLabelBinarizer()
category_encoded = mlb_1.fit_transform(df["Category"])
category_df = pd.DataFrame(category_encoded, columns=mlb_1.classes_)

# Define which base features to use from the original DataFrame
feature_cols = [
    "Brightness",
    "Contrast",
    "Saturation",
    "Hue_Entropy",
    "Hue",
    "Sat",
    "Val",
]

# Create input features (X) by combining encoded categories, base features, RGB, and histograms
X = pd.concat([category_df, df[feature_cols], rgb_flat_df, hist_feat_df], axis=1)

# Create target labels (y) from the same base features, RGB, and histograms
y = pd.concat([df[feature_cols], rgb_flat_df, hist_feat_df], axis=1)

# Merge X and y into a single DataFrame to ensure any rows with NaNs are dropped uniformly
combined = pd.concat([X, y], axis=1)
combined.dropna(inplace=True)  # Drop rows with missing values

# Split back the cleaned features and labels
X_clean = combined.iloc[:, : X.shape[1]]
y_clean = combined.iloc[:, X.shape[1] :]

# Define a variety of regression models to test
base_models = {
    "RandomForest": RandomForestRegressor(),
    "GradientBoosting": GradientBoostingRegressor(),
    "MLP": MLPRegressor(max_iter=1000),
    "SVR": SVR(),
    "XGBoost": XGBRegressor(),
    "LightGBM": LGBMRegressor(),
    "BayesianRidge": BayesianRidge(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
}

# Train each model using a MultiOutputRegressor wrapper (handles multi-target regression)
trained_models_1 = {}
for name, model in base_models.items():
    multi_output_model = MultiOutputRegressor(model)
    multi_output_model.fit(X_clean, y_clean)
    trained_models_1[name] = multi_output_model  # Save the trained model

print("All models trained successfully.")


In [ ]:
# Compute the average brightness of the image by converting it to grayscale
def get_brightness(image):
    return np.mean(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY))


# Calculate contrast using histogram equalization and standard deviation of pixel values
def get_contrast(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return exposure.equalize_hist(gray).std()


# Calculate the average saturation from the HSV representation of the image
def get_saturation(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return np.mean(hsv[:, :, 1])


# Compute a 3D color histogram and return only the first 10 values as features
def get_color_histogram(image):
    hist = cv2.calcHist([image], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
    return hist.flatten()[:10]


# Estimate color coherence via entropy in the hue channel
def get_color_coherence_entropy(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hue_channel = hsv[:, :, 0]
    hist = cv2.calcHist([hue_channel], [0], None, [180], [0, 180])
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-6))  # Add small value to avoid log(0)
    return entropy


# Extract mean hue, saturation, and value from HSV representation
def get_hsv_analysis(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return np.mean(hsv[:, :, 0]), np.mean(hsv[:, :, 1]), np.mean(hsv[:, :, 2])


# Preprocess xkcd color dictionary to use for closest color matching
xkcd_colors = {
    name.replace("xkcd:", "").title(): mcolors.to_rgb(code)
    for name, code in mcolors.XKCD_COLORS.items()
}


# Find the closest xkcd color label to an RGB value using Euclidean distance
def closest_xkcd_color(rgb):
    r, g, b = np.array(rgb) / 255  # Normalize RGB to 0-1
    min_dist = float("inf")
    closest_name = None
    for name, (xr, xg, xb) in xkcd_colors.items():
        dist = np.sqrt((r - xr) ** 2 + (g - xg) ** 2 + (b - xb) ** 2)
        if dist < min_dist:
            min_dist = dist
            closest_name = name
    return closest_name


# Extract top k dominant colors using KMeans clustering and label them
def get_dominant_colors(image, k=3):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    pixels = image.reshape(-1, 3)
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3)
    kmeans.fit(pixels)
    counter = Counter(kmeans.labels_)

    top_colors = []
    for i, _ in counter.most_common(k):
        rgb = kmeans.cluster_centers_[i].astype(int).tolist()
        color_label = closest_xkcd_color(rgb)
        top_colors.append({"rgb": rgb, "label": color_label})
    return top_colors


# Combine all feature extraction steps for a single image
def extract_features_from_image_1(img_path):
    image = cv2.imread(img_path)
    if image is None:
        raise ValueError(f"Unable to load image from path: {img_path}")

    # Extract basic features
    brightness = get_brightness(image)
    contrast = get_contrast(image)
    saturation = get_saturation(image)
    hue_entropy = get_color_coherence_entropy(image)
    hue, sat, val = get_hsv_analysis(image)

    # Extract color features
    rgb = get_dominant_colors(image)
    rgb_flat = flatten_rgb([c["rgb"] for c in rgb])  # Flatten RGB lists into 1D
    hist_feat = get_color_histogram(image).tolist()

    return (
        [brightness, contrast, saturation, hue_entropy, hue, sat, val],
        rgb_flat[:9],  # Use only first 9 values (3 colors × 3 channels)
        hist_feat,
    )


# Analyze text properties within the image using OCR
def analyze_text(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = image.shape[:2]

    # Use Tesseract to extract word-level metadata
    d = pytesseract.image_to_data(
        gray, output_type=pytesseract.Output.DICT, config="--psm 6"
    )

    word_count = 0
    font_sizes = []
    text_positions = []
    font_colors = []
    font_styles = []
    valid_text = []

    for i in range(len(d["text"])):
        text = d["text"][i].strip()
        if text and len(text) >= 3 and text.isalnum():  # Filter short/noisy text
            word_count += 1
            font_sizes.append(d["height"][i])

            # Determine text vertical position in the image
            y_center = (d["top"][i] + d["height"][i]) / 2
            if y_center < h * 0.33:
                text_positions.append("Top")
            elif y_center < h * 0.66:
                text_positions.append("Middle")
            else:
                text_positions.append("Bottom")

            # Estimate average font color from bounding box
            x, y, w_box, h_box = (
                d["left"][i],
                d["top"][i],
                d["width"][i],
                d["height"][i],
            )
            roi = image[y : y + h_box, x : x + w_box]
            if roi.size > 0:
                avg_color = cv2.mean(roi)[:3]
                font_colors.append(avg_color)

            # Optional: extract style info (if available from OCR)
            style = []
            font_attr = d.get("font_attrs", [""] * len(d["text"]))[i]
            if "bold" in font_attr:
                style.append("Bold")
            if "italic" in font_attr:
                style.append("Italic")
            if "serif" in font_attr:
                style.append("Serif")
            if "sans" in font_attr:
                style.append("Sans-serif")
            font_styles.append(", ".join(style) if style else "Regular")

            valid_text.append(text)

    # Aggregate text features
    avg_color = np.mean(font_colors, axis=0) if font_colors else (0, 0, 0)
    position = (
        Counter(text_positions).most_common(1)[0][0] if text_positions else "Unknown"
    )
    most_common_style = (
        Counter(font_styles).most_common(1)[0][0] if font_styles else "Regular"
    )

    return {
        "Word_Count": word_count,
        "Average_Font_Size": np.mean(font_sizes) if font_sizes else 0,
        "Text_Position": position,
        "Font_Color": tuple(map(int, avg_color)),
        "Font_Style": most_common_style,
        "Text": " ".join(valid_text),
    }


# Compute a basic saliency map using Sobel edge detection
def compute_saliency(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    saliency = sobel(gray)
    return saliency


# Use Canny edge detection to estimate contrast via edge strength
def analyze_contrast(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return np.mean(edges)


# Combine saliency and contrast analysis into one feature dict
def extract_image_features_3(image_path):
    saliency = compute_saliency(image_path)
    contrast = analyze_contrast(image_path)

    return {
        "Saliency": float(np.mean(saliency)) if saliency is not None else 0.0,
        "Contrast": float(contrast) if contrast is not None else 0.0,
    }

In [ ]:
# Load the text-based features extracted from thumbnails
text_df = pd.read_csv("/kaggle/input/thumbnail-features/thumbnail_text_feature.csv")

# One-hot encode the 'Category' column (multi-label format)
mlb_2 = MultiLabelBinarizer()
category_encoded = mlb_2.fit_transform(text_df["Category"])
category_df = pd.DataFrame(category_encoded, columns=mlb_2.classes_)

# Split the 'Font_Color' column into separate RGB columns
text_df[["Font_Color_R", "Font_Color_G", "Font_Color_B"]] = pd.DataFrame(
    text_df["Font_Color"]
    .apply(lambda x: eval(x) if isinstance(x, str) else (0, 0, 0))
    .tolist(),
    index=text_df.index,
)

# Initialize label encoders for categorical string fields
position_encoder = LabelEncoder()
style_encoder = LabelEncoder()

# Fit encoders on unique values from the respective columns
position_encoder.fit(text_df["Text_Position"].unique())
style_encoder.fit(text_df["Font_Style"].unique())

# Transform the string values to integer-encoded representations
text_df["Text_Position_Encoded"] = position_encoder.transform(text_df["Text_Position"])
text_df["Font_Style_Encoded"] = style_encoder.transform(text_df["Font_Style"])

# Prepare the target variable `y` (what we want to predict)
y = text_df[
    ["Word_Count", "Average_Font_Size", "Text_Position_Encoded", "Font_Style_Encoded"]
    + ["Font_Color_R", "Font_Color_G", "Font_Color_B"]
]

# Prepare the input features `X` by combining encoded categories and other numerical features
X = pd.concat(
    [
        category_df,
        text_df[
            [
                "Word_Count",
                "Average_Font_Size",
                "Text_Position_Encoded",
                "Font_Style_Encoded",
                "Font_Color_R",
                "Font_Color_G",
                "Font_Color_B",
            ]
        ],
    ],
    axis=1,
)

# Define a set of regression models wrapped in MultiOutputRegressor to handle multiple targets
models_dict_2 = {
    "RandomForest": MultiOutputRegressor(RandomForestRegressor()),
    "GradientBoosting": MultiOutputRegressor(GradientBoostingRegressor()),
    "SVR": MultiOutputRegressor(SVR(kernel="rbf", C=100, epsilon=0.1)),
    "KNN": MultiOutputRegressor(KNeighborsRegressor(n_neighbors=5)),
    "DecisionTree": MultiOutputRegressor(DecisionTreeRegressor()),
    "AdaBoost": MultiOutputRegressor(AdaBoostRegressor(n_estimators=100)),
    "Lasso": MultiOutputRegressor(Lasso(alpha=0.01)),
    "ElasticNet": MultiOutputRegressor(ElasticNet(alpha=0.01, l1_ratio=0.5)),
    "Ridge": MultiOutputRegressor(Ridge(alpha=1.0)),
    "MLP": MultiOutputRegressor(
        MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000)
    ),
}

# Train all the models on the feature set `X` and target `y`
for name, model in models_dict_2.items():
    model.fit(X, y)

print("All models trained successfully.")

In [ ]:
# Load composition features (e.g., saliency, contrast) from CSV
df_saliency_contrast = pd.read_csv(
    "/kaggle/input/thumbnail-features/thumbnail_composition_feature.csv"
)

# One-hot encode the 'Category' column
# Since this dataset seems to contain only one category per row, wrap the value in a list
mlb_3 = MultiLabelBinarizer()
category_encoded = mlb_3.fit_transform(
    df_saliency_contrast["Category"].apply(lambda x: [x])
)
category_df = pd.DataFrame(category_encoded, columns=mlb_3.classes_)

# Input features consist of only the one-hot encoded categories
X = pd.concat([category_df], axis=1)

# Target variables are 'Saliency' and 'Contrast'
y = df_saliency_contrast[["Saliency", "Contrast"]]

# Define a suite of regression models, each wrapped with MultiOutputRegressor to predict both targets
models_dict_3 = {
    "RandomForest": MultiOutputRegressor(RandomForestRegressor()),
    "GradientBoosting": MultiOutputRegressor(GradientBoostingRegressor()),
    "SVR": MultiOutputRegressor(SVR(kernel="rbf", C=100, epsilon=0.1)),
    "KNN": MultiOutputRegressor(KNeighborsRegressor(n_neighbors=5)),
    "DecisionTree": MultiOutputRegressor(DecisionTreeRegressor()),
    "AdaBoost": MultiOutputRegressor(AdaBoostRegressor(n_estimators=100)),
    "Lasso": MultiOutputRegressor(Lasso(alpha=0.01)),
    "ElasticNet": MultiOutputRegressor(ElasticNet(alpha=0.01, l1_ratio=0.5)),
    "Ridge": MultiOutputRegressor(Ridge(alpha=1.0)),
    "MLP": MultiOutputRegressor(
        MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000)
    ),
}

# Train each model on the input features `X` and target variables `y`
for name, model in models_dict_3.items():
    model.fit(X, y)

print("All models trained successfully.")

In [ ]:
# Suggest improvements for visual features based on a trained model (set 1)
def suggest_improvements_for_1(
    image_features, category_input, model, rgb_flat, hist_feat
):
    # Unpack visual features
    brightness, contrast, saturation, hue_entropy, hue, sat, val = image_features

    # Encode the category input
    category_vec = mlb_1.transform([category_input])
    feature_input = pd.DataFrame(category_vec, columns=mlb_1.classes_)

    # Create individual feature DataFrames
    additional_feats = pd.DataFrame(
        [[brightness, contrast, saturation, hue_entropy, hue, sat, val]],
        columns=feature_cols,
    )
    rgb_feats = pd.DataFrame([rgb_flat], columns=[f"DRGB_{i}" for i in range(9)])
    hist_feats = pd.DataFrame([hist_feat], columns=[f"Hist_{i}" for i in range(10)])

    # Combine all features into a single input row
    full_input = pd.concat(
        [feature_input, additional_feats, rgb_feats, hist_feats], axis=1
    )

    # Predict the ideal feature values
    prediction = model.predict(full_input)[0]

    # Combine the actual input features for comparison
    actual = np.concatenate(
        [additional_feats.values[0], rgb_feats.values[0], hist_feats.values[0]]
    )
    feature_names = y_clean.columns

    # Generate suggestions based on prediction vs actual values
    suggestions = []
    for i, feature_name in enumerate(feature_names):
        actual_val = actual[i]
        predicted_val = prediction[i]
        diff = predicted_val - actual_val

        if abs(diff) < 0.01:
            suggestion = "No change"
        elif diff > 0:
            suggestion = f"Increase by {round(diff, 2)}"
        else:
            suggestion = f"Decrease by {round(abs(diff), 2)}"

        suggestions.append(
            (feature_name, round(actual_val, 2), round(predicted_val, 2), suggestion)
        )

    return suggestions


# Suggest improvements for text-based features (set 2)
def suggest_improvements_2(
    test_image_features, category_input, model, category_df, mlb
):
    # Encode the input category
    category_vec = mlb_2.transform([category_input])
    feature_input = pd.DataFrame(category_vec, columns=mlb_2.classes_)

    # Prepare other text-related features from the test input
    additional_feats = pd.DataFrame(
        [
            [
                test_image_features["Word_Count"],
                test_image_features["Average_Font_Size"],
                test_image_features["Text_Position_Encoded"],
                test_image_features["Font_Style_Encoded"],
                *test_image_features["Font_Color"],
            ]
        ],
        columns=[
            "Word_Count",
            "Average_Font_Size",
            "Text_Position_Encoded",
            "Font_Style_Encoded",
            "Font_Color_R",
            "Font_Color_G",
            "Font_Color_B",
        ],
    )

    # Combine all input features
    full_input = pd.concat([feature_input, additional_feats], axis=1)

    # Predict the ideal values
    prediction = model.predict(full_input)[0]

    # Compare with actual values to generate improvement suggestions
    suggestions = []
    actual_values = additional_feats.iloc[0, :].values
    feature_names = additional_feats.columns

    for feature_name, actual_val, predicted_val in zip(
        feature_names, actual_values, prediction
    ):
        diff = predicted_val - actual_val
        if abs(diff) < 0.01:
            suggestion = "No change"
        elif diff > 0:
            suggestion = f"Increase by {round(diff, 2)}"
        else:
            suggestion = f"Decrease by {round(abs(diff), 2)}"
        suggestions.append(
            (feature_name, round(actual_val, 2), round(predicted_val, 2), suggestion)
        )

    return suggestions


# Suggest improvements for composition features (set 3)
def suggest_improvements_3(test_features, category_input, model):
    # Encode the input category
    category_vec = mlb_3.transform([[category_input]])
    input_df = pd.DataFrame(category_vec, columns=mlb_3.classes_)

    # Predict the ideal saliency and contrast values
    prediction = model.predict(input_df)[0]

    suggestions = []

    # Compare predicted vs actual to suggest adjustments
    for i, feature_name in enumerate(test_features.keys()):
        actual_val = test_features[feature_name]
        predicted_val = prediction[i]
        diff = predicted_val - actual_val

        if abs(diff) < 0.01:
            suggestion = "No change"
        elif diff > 0:
            suggestion = f"Increase by {round(diff, 3)}"
        else:
            suggestion = f"Decrease by {round(abs(diff), 3)}"

        suggestions.append(
            (feature_name, round(actual_val, 3), round(predicted_val, 3), suggestion)
        )

    return suggestions

In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract
!pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

In [ ]:
# Install required libraries: Transformers for model loading, Accelerate for performance, and BitsAndBytes for 4-bit quantization
!pip install transformers accelerate bitsandbytes --quiet

# Import required libraries
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
else:
    raise ValueError("Missing Hugging Face token. Set HF_TOKEN environment variable.")


# Specify the name of the model to load from Hugging Face
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# Set up quantization config for loading the model in 4-bit precision to reduce memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load model weights in 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # Use float16 for intermediate computations
    bnb_4bit_use_double_quant=True,  # Use nested quantization for better accuracy
)

# Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure there is a pad token defined; if not, default to end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the pre-trained language model with quantization and automatic device placement (CPU/GPU)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically assign model parts to available devices (like multiple GPUs)
)

In [ ]:
def clean_llm_response(prompt, response):
    """Remove prompt-like content from the beginning of the LLM response."""
    response = response.strip()
    prompt_lines = [
        line.strip() for line in prompt.strip().splitlines() if line.strip()
    ]

    for line in prompt_lines:
        if response.startswith(line):
            response = response[len(line) :].lstrip()

    return response

In [ ]:
# Generate visual feature enhancement suggestions using KNN model and summarize with a language model
def send_visual_suggestions_to_llm(
    img_path, input_categories, trained_models, tokenizer, model
):
    # Extract image features (visual stats, RGB, histogram) from the thumbnail
    features, rgb_flat, hist_feat = extract_features_from_image_1(img_path)

    # Find the KNN model among the trained models
    knn_model = None
    for model_name, model_inst in trained_models.items():
        if "knn" in model_name.lower():
            knn_model = model_inst
            break

    # If no KNN model is found, return an error message
    if knn_model is None:
        return "KNN model not found in trained_models."

    # Prepare a string to summarize feature suggestions from the KNN model
    full_feedback = "\nVisual Feature Suggestions from KNN Model:\n"
    full_feedback += "=" * 60 + "\n"

    # Generate improvement suggestions using the KNN model
    improvement_suggestions = suggest_improvements_for_1(
        features, input_categories, knn_model, rgb_flat, hist_feat
    )

    # Format suggestions into readable text
    for feature, actual, predicted, suggestion in improvement_suggestions:
        full_feedback += (
            f"{feature}: Actual = {actual:.2f}, Predicted = {predicted:.2f}, "
            f"Suggestion = {suggestion}\n"
        )
    full_feedback += "-" * 50 + "\n"

    # Construct a prompt to send to the language model, instructing it to analyze and explain the suggestions
    prompt = f"""
You are a helpful and skilled YouTube visual content advisor.

A KNN machine learning model has analyzed various **visual features** of a YouTube thumbnail, including:
- Brightness
- Contrast
- Saturation
- Hue Entropy (color complexity)
- Hue (dominant color tone)
- Value (overall brightness level)
- RGB color distribution
- Color histogram bins
Bin 0 maps to black/dark grey.
Bin 1 maps to Dark Blue
Bin 2 maps to Dark Green
Bin 3 maps to Dark Red
Bin 4 maps to Brown
Bin 5 maps to Yellow
Bin 6 maps to Mint
Bin 7 maps to Sky Blue
Bin 8 maps to White

The model has suggested specific improvements based on the differences between the current values and predicted target values.

Your task is to:
1. Summarize the **key visual issues** from the analysis
2. Provide **friendly, actionable suggestions** on how to fix or enhance these visual elements
3. Explain why suggestions could improve the thumbnail's performance (e.g., better click-through rate, clarity, or emotional impact)

Use simple language, emojis, and bullet points for clarity. 
---
{full_feedback}
---

Please break your response into:
- An intro on visual attributes
- Bullet points with visual advice
- A closing tip for creators
""".strip()

    # Tokenize the prompt for the LLM
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, padding=True, max_length=100000
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Generate a response from the language model without gradient tracking
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1000000,  # Generate a long response if needed
            temperature=0.7,  # Sampling temperature for creativity
            top_p=0.9,  # Nucleus sampling for coherence
            repetition_penalty=1.1,  # Slight penalty to reduce repetitiveness
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode the generated output
    raw_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean up and return the model's formatted advice
    return clean_llm_response(prompt, raw_response)

In [ ]:
# Generate text overlay improvement suggestions from a KNN model and summarize them using a language model
def send_text_suggestions_to_llm(
    img_path,
    category_input,
    models_dict_2,
    position_encoder,
    style_encoder,
    category_df,
    mlb,
    tokenizer,
    model,
):
    # Step 1: Extract text features using OCR from the image
    test_image_features = analyze_text(img_path)
    if not test_image_features:
        return "No text features were extracted from the image."

    # Step 2: Encode categorical text position and font style for the model
    test_image_features["Text_Position_Encoded"] = position_encoder.transform(
        [test_image_features["Text_Position"]]
    )[0]
    test_image_features["Font_Style_Encoded"] = style_encoder.transform(
        [test_image_features["Font_Style"]]
    )[0]

    # Step 3: Locate the KNN model for text features
    knn_model = None
    for model_name, model_inst in models_dict_2.items():
        if "knn" in model_name.lower():
            knn_model = model_inst
            break

    if knn_model is None:
        return "KNN model not found in models_dict_2."

    # Step 4: Generate raw improvement suggestions using the KNN model
    full_feedback = "\nText Overlay Suggestions (KNN Model Only):\n"
    full_feedback += "=" * 60 + "\n"

    improvement_suggestions = suggest_improvements_2(
        test_image_features, category_input, knn_model, category_df, mlb
    )

    # Step 5: Format suggestions for readability
    for feature, actual, predicted, suggestion in improvement_suggestions:
        full_feedback += (
            f"{feature}: Actual = {actual} → Predicted: {predicted} → {suggestion}\n"
        )
    full_feedback += "-" * 50 + "\n"

    # Step 6: Create a prompt for the language model to explain and suggest design changes
    prompt = f"""
You are a helpful AI design assistant specializing in YouTube thumbnails.

You have just received feedback from a KNN machine learning model that analyzed the **text overlay** in a thumbnail. It evaluated elements like:
- Word count
- Font size
- Font style
- Text position

Your task is to:
1. Summarize the **key text issues** from the analysis
2. Provide **friendly, actionable suggestions** on how to fix or enhance these text elements
3. Explain why suggestions could improve the thumbnail's performance (e.g., better click-through rate, clarity, or emotional impact)

Use simple language, emojis, and bullet points for clarity. 

---
{full_feedback}
---

Please break your response into:
- An intro on text attributes
- Bullet points with advice
- A closing tip for creators
""".strip()

    # Step 7: Tokenize the prompt for input to the language model
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, padding=True, max_length=10000000
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Step 8: Generate a response from the language model (LLM)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1000000,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Step 9: Decode the response and clean it if needed
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in full_response:
        return full_response.split("Answer:")[-1].strip()

    raw_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_llm_response(prompt, raw_response)

In [ ]:
# Generate suggestions for improving saliency and contrast in a thumbnail using a KNN model and explain them using an LLM
def send_saliency_contrast_suggestions_to_llm(
    image_path, category_input, models_dict_3, tokenizer, model
):
    # Step 1: Extract saliency and contrast features from the image
    test_features = extract_image_features_3(image_path)

    # Step 2: Identify the KNN model from the provided model dictionary
    knn_model = None
    for model_name, model_inst in models_dict_3.items():
        if "knn" in model_name.lower():
            knn_model = model_inst
            break

    # Step 3: Return an error message if no KNN model is found
    if knn_model is None:
        return "KNN model not found in models_dict_3."

    # Step 4: Collect improvement suggestions from the KNN model
    full_feedback = "\nSaliency & Contrast Suggestions (KNN Model Only):\n"
    full_feedback += "=" * 60 + "\n"
    improvements = suggest_improvements_3(test_features, category_input, knn_model)

    # Step 5: Format the model's predictions and suggestions into readable feedback
    for feature, actual, predicted, suggestion in improvements:
        full_feedback += (
            f"{feature}: Actual = {actual:.2f}, Predicted = {predicted:.2f}, "
            f"Suggestion = {suggestion}\n"
        )
    full_feedback += "-" * 50 + "\n"

    # Step 6: Construct a prompt for the language model to turn raw suggestions into creative design advice
    prompt = f"""
You're an expert in visual clarity and user attention on digital platforms.

The following data is from a **KNN model** analyzing a YouTube thumbnail’s visual composition — specifically **saliency** (what draws the eye) and **contrast** (visual clarity and separation). Your role is to:
1. Interpret these insights simply and clearly
2. Give creative, practical suggestions to improve the image
3. Educate the user on *why* these changes matter (click-through rate, readability, visual hierarchy, etc.)

Use simple language, emojis, and bullet points for clarity. 

---
{full_feedback}
---

Please respond with:
Please break your response into:
- An intro on composition 
- Bullet points with key advice
- A closing tip for creators
""".strip()

    # Step 7: Tokenize the prompt and prepare for input to the LLM
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, padding=True, max_length=10000000
    )
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    # Step 8: Generate a response using the language model
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1000000,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            num_return_sequences=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Step 9: Decode and return the model’s response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # If the model returns a response prefixed with "Answer:", extract the relevant part
    if "Answer:" in full_response:
        return full_response.split("Answer:")[-1].strip()

    # Otherwise, clean and return the full response
    raw_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_llm_response(prompt, raw_response)

In [ ]:
!pip install gradio --quiet

In [ ]:
# Import required libraries
import gradio as gr
from PIL import Image
import os

# Predefined list of categories for user to choose from
category_list = [
    "Science",
    "News",
    "Food",
    "Blog",
    "Tech",
    "Informative",
    "Comedy,Entertainment",
    "Automobile",
    "Tech,Informative",
    "Automobile,Comedy",
    "VideoGames",
    "Food,Entertainment",
    "Blog,Comedy",
    "Comedy,Informative",
    "Tech,Comedy",
    "Comedy",
    "Blog,Science",
    "Blog,Entertainment",
    "Entertainment,Comedy",
    "Tech,News",
    "Entertainment,Blog",
]


# Main analysis function that runs all three LLM-enhanced analyses
def analyze_thumbnail_split(image: Image.Image, category: str):
    img_path = "/tmp/uploaded_thumbnail.png"
    image.save(img_path)

    # Try visual analysis
    try:
        visual_feedback = send_visual_suggestions_to_llm(
            img_path, [category], trained_models_1, tokenizer, model
        )
    except Exception as e:
        visual_feedback = f"Visual Analysis Failed: {str(e)}"

    # Try text overlay analysis
    try:
        text_feedback = send_text_suggestions_to_llm(
            img_path,
            category,
            models_dict_2,
            position_encoder,
            style_encoder,
            category_df,
            mlb_2,
            tokenizer,
            model,
        )
    except Exception as e:
        text_feedback = f"Text Overlay Analysis Failed: {str(e)}"

    # Try composition analysis (saliency & contrast)
    try:
        composition_feedback = send_saliency_contrast_suggestions_to_llm(
            img_path, category, models_dict_3, tokenizer, model
        )
    except Exception as e:
        composition_feedback = f"Composition Analysis Failed: {str(e)}"

    return visual_feedback, text_feedback, composition_feedback


# Gradio UI definition
with gr.Blocks(title="Smart Thumbnail Analyzer") as demo:
    # Title & intro
    gr.Markdown(
        """
    <h1 style='text-align: center; font-size: 2.5em;'>🧠 Smart Thumbnail Analyzer</h1>
    <p style='text-align: center;'>Upload a thumbnail and choose a category to get AI-powered suggestions.</p>
    """,
        elem_id="centered-heading",
    )

    # Upload section with image and category dropdown
    with gr.Row():
        gr.Column(scale=1)  # Spacer column

        with gr.Column(scale=2, min_width=400):
            image_input = gr.Image(type="pil", label="📷 Upload Thumbnail")
            category_input = gr.Dropdown(
                choices=category_list, label="🎯 Select Thumbnail Category"
            )

        gr.Column(scale=1)  # Spacer column

    submit_btn = gr.Button("Analyze Thumbnail ✅")

    # Tabbed output display for visual, text, and composition feedback
    with gr.Tabs():
        with gr.TabItem("🎨 Visual Features"):
            visual_output = gr.Markdown(label="Visual Feature Feedback")

        with gr.TabItem("🔤 Text Overlay"):
            text_output = gr.Markdown(label="Text Overlay Suggestions")

        with gr.TabItem("🖼️ Composition"):
            composition_output = gr.Markdown(label="Saliency & Contrast Suggestions")

    status = gr.Markdown("")

    # Step-by-step yield-based feedback generator for better UX
    def analyze_with_status(image, category):
        img_path = "/tmp/uploaded_thumbnail.png"
        image.save(img_path)

        # Step 1: Indicate start of analysis
        yield "", "", "", "<br><br>⏳ Starting thumbnail analysis..."

        # Step 2: Visual feature analysis
        yield "<br><br>⏳ Generating visual feedback...", "", "", ""
        try:
            visual_feedback = send_visual_suggestions_to_llm(
                img_path, [category], trained_models_1, tokenizer, model
            )
        except Exception as e:
            visual_feedback = f"❌ Visual Analysis Failed: {str(e)}"
        yield visual_feedback, "", "", ""

        # Step 3: Text overlay feedback
        yield visual_feedback, "<br><br>⏳ Generating text overlay suggestions...", "", ""
        try:
            text_feedback = send_text_suggestions_to_llm(
                img_path,
                category,
                models_dict_2,
                position_encoder,
                style_encoder,
                category_df,
                mlb_2,
                tokenizer,
                model,
            )
        except Exception as e:
            text_feedback = f"❌ Text Overlay Analysis Failed: {str(e)}"
        yield visual_feedback, text_feedback, "", ""

        # Step 4: Saliency and contrast (composition)
        yield visual_feedback, text_feedback, "<br><br>⏳ Analyzing saliency & contrast...", ""
        try:
            composition_feedback = send_saliency_contrast_suggestions_to_llm(
                img_path, category, models_dict_3, tokenizer, model
            )
        except Exception as e:
            composition_feedback = f"❌ Composition Analysis Failed: {str(e)}"

        # Final yield with all feedback displayed
        yield visual_feedback, text_feedback, composition_feedback, ""

        # Clean up temporary file
        try:
            os.remove(img_path)
        except Exception as e:
            print(f"⚠️ Could not delete temp image: {e}")

    # Connect button click to multi-step analysis function
    submit_btn.click(
        analyze_with_status,
        inputs=[image_input, category_input],
        outputs=[visual_output, text_output, composition_output, status],
    )

# Launch the app
demo.launch()

In [ ]:
try:
    os.remove("/tmp/uploaded_thumbnail.png")
except Exception as e:
    print(f"⚠️ Could not delete temp image: {e}")